# NYCタクシー乗車データ分析 — v020-exp-23

## 分析計画（取得前）
Kaggle参照: https://www.kaggle.com/datasets?search=nyc+taxi+trips

1. Kaggle SDKで距離・乗降日時・料金・チップ・支払方式を持つ公開データを検索し、ファイル一覧とサイズを確認して取得する。失敗時のみ指定CSVへ切り替える。
2. 行数・欠損・重複・期間・分布を確認し、データ定義と分析前提manifestを記録する。
3. 非正の距離/時間、負額、不自然な速度、料金構成の不整合を意味的に検査する。元データは保持し、除外を明記する。
4. 分布図・密度図・相関図・支払方式/距離帯別集計を選択し、PearsonとSpearman、チップ金額と率を区別する。fareとtotalの違い、および現金チップの未観測を考慮する。
5. 妥当な閾値・サブセットで感度を測り、因果効果は主張しない。
6. 結果を再読し最大3回の自然言語反復依頼を原文記録して実行する。全コードセルを上から再実行後、根拠とvisual_audit、lifecycle状態を保存する。

実行はJupyter MCPのみ。トークンを出力・保存しない。

In [4]:
from pathlib import Path
import os, sys, json, hashlib, io, contextlib, traceback
from datetime import datetime, timezone
from dataclasses import asdict
import nbformat
import pandas as pd
import numpy as np
from IPython.display import display
repo = Path('/home/nahisaho/GitHub/jupytermind')
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
from ai_data_scientist import project_manager, language_router, lifecycle
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-23-taxis')
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
run_id = 'v020-exp-23'
lifecycle.register_run(run_id, handle.notebook_path)
language = language_router.detect_language('タクシー乗車データの距離・所要時間・料金・チップの関係を調べてください。')
phase_log = []
@contextlib.contextmanager
def phase(name, writing=False):
    if lifecycle.is_cancel_requested(run_id):
        raise RuntimeError('実行のキャンセルが要求されています。')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(run_id)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(run_id)
        phase_log.append({'phase': name, 'event': 'failed', 'utc': datetime.now(timezone.utc).isoformat()})
        raise
    finally:
        end(run_id)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
def emit(value):
    display({'text/plain': json.dumps(value, ensure_ascii=False, indent=2, default=str)}, raw=True)
with phase('準備'):
    emit({'run_id': run_id, 'language': language, 'notebook': str(handle.notebook_path), 'execution_route': 'Jupyter MCPのみ', 'status': asdict(lifecycle.get_run_status(run_id))})


{
  "run_id": "v020-exp-23",
  "language": "ja",
  "notebook": "/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-23-taxis/notebooks/kaggle-v020-kaggle-exp-23-taxis.ipynb",
  "execution_route": "Jupyter MCPのみ",
  "status": {
    "run_id": "v020-exp-23",
    "state": "running",
    "active_cell_executions": 1,
    "pending_notebook_writes": 0,
    "locks_held": 0,
    "notebook_path": "/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-23-taxis/notebooks/kaggle-v020-kaggle-exp-23-taxis.ipynb",
    "reason": null
  }
}

## Kaggle公開データ検索
認証失敗時は操作・例外型・HTTP状態を記録する。秘密漏洩を防ぐため例外本文は保存しない。

In [5]:
from kaggle.api.kaggle_api_extended import KaggleApi
import inspect
kaggle_log = []
search_results = []
file_catalog = []
api_ready = False
# 認証処理の出力・例外本文は保存しない。認証情報を出力しないための境界。
def api_failure(stage, exc):
    code = getattr(exc, 'status', None) or getattr(exc, 'status_code', None)
    if code is None:
        code = getattr(getattr(exc, 'response', None), 'status_code', None)
    return {'stage': stage, 'exception_type': type(exc).__name__, 'http_status': code,
            'detail': '例外本文は認証情報漏洩防止のため非保存。型とHTTP状態を記録。'}
with phase('Kaggle認証・公開データ検索'):
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            KaggleApi().authenticate()
            api = KaggleApi()
            api.authenticate()
        api_ready = True
        kaggle_log.append({'stage': 'authenticate', 'status': 'success'})
    except (Exception, SystemExit) as exc:
        kaggle_log.append(api_failure('authenticate', exc))
    if api_ready:
        for query in ['nyc taxi trips', 'seaborn taxis', 'taxi trips']:
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    hits = api.dataset_list(search=query, page=1)
                for d in (hits or []):
                    search_results.append({'query': query, 'slug': getattr(d, 'ref', None),
                        'title': getattr(d, 'title', None), 'total_bytes': getattr(d, 'total_bytes', None)})
                kaggle_log.append({'stage': 'dataset_list', 'query': query, 'status': 'success', 'count': len(hits or [])})
            except Exception as exc:
                kaggle_log.append(api_failure('dataset_list: ' + query, exc))
    emit({'kaggle_log': kaggle_log, 'search_results': search_results})


{
  "kaggle_log": [
    {
      "stage": "authenticate",
      "status": "success"
    },
    {
      "stage": "dataset_list",
      "query": "nyc taxi trips",
      "status": "success",
      "count": 20
    },
    {
      "stage": "dataset_list",
      "query": "seaborn taxis",
      "status": "success",
      "count": 2
    },
    {
      "stage": "dataset_list",
      "query": "taxi trips",
      "status": "success",
      "count": 20
    }
  ],
  "search_results": [
    {
      "query": "nyc taxi trips",
      "slug": "fivethirtyeight/uber-pickups-in-new-york-city",
      "title": "Uber Pickups in New York City",
      "total_bytes": 114370464
    },
    {
      "query": "nyc taxi trips",
      "slug": "shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021",
      "title": "Uber NYC for-hire vehicles trip data (2021)",
      "total_bytes": 4539471170
    },
    {
      "query": "nyc taxi trips",
      "slug": "elemento/nyc-yellow-taxi-trip-data",
      "title": "NYC Yellow Taxi Trip 

## 候補選定
検索には数GBのTLCデータや乗車位置のみのデータが含まれる。まず全Seabornデータ掲載版・cleaned版・NYCサンプルのファイル一覧を比較する。未加工に近いtaxis.csvが得られれば、旧実験に相当する小規模な全件分析を優先する。これは便宜標本でありNYC全体への代表性は保証しない。

In [8]:
with phase('Kaggle候補ファイル一覧'):
    if api_ready:
        for slug in ['abdoomoh/all-seaborn-built-in-datasets', 'abdmental01/taxis-dataset-yellow-taxi', 'anandaramg/taxi-trip-data-nyc']:
            if not any(r['slug'] == slug for r in search_results):
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    catalog = api.dataset_list_files(slug, page_size=100)
                files = getattr(catalog, 'files', None) or getattr(catalog, 'dataset_files', None) or []
                for item in files:
                    file_catalog.append({'slug': slug, 'name': getattr(item, 'name', None),
                                         'size': getattr(item, 'total_bytes', None)})
                kaggle_log.append({'stage': 'dataset_list_files', 'slug': slug, 'status': 'success', 'file_count': len(files)})
                if not files:
                    emit({'catalog_type': type(catalog).__name__, 'public_attributes': [a for a in dir(catalog) if not a.startswith('_')]})
            except Exception as exc:
                kaggle_log.append(api_failure('dataset_list_files: ' + slug, exc))
    emit({'file_catalog': file_catalog, 'kaggle_log': kaggle_log})


{
  "file_catalog": [
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/anagrams.csv",
      "size": 420
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/anscombe.csv",
      "size": 601
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/attention.csv",
      "size": 1269
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/brain_networks.csv",
      "size": 1076925
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/car_crashes.csv",
      "size": 2957
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": "Seaborn All Built-in Datasets/diamonds.csv",
      "size": 2625846
    },
    {
      "slug": "abdoomoh/all-seaborn-built-in-datasets",
      "name": 